# T39 - 成交数据
## 第七章：报告生成

本章介绍成交数据分析报告的生成，包括：n1. 数据概览报告
2. 流动性分析报告
3. 结构分析报告
4. 综合分析报告
5. 报告导出

## 1. 导入必要的库

In [ ]:
# 数据处理
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import logging
import json
import os
from typing import Dict, List, Optional

# 数据库连接
import sqlalchemy

# 导入配置
from config import config

# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger('报告生成')

print('导入成功！')

## 2. 加载数据

In [ ]:
def load_trade_data(days: int = 60) -> pd.DataFrame:
    """加载成交数据"""
    engine = sqlalchemy.create_engine(
        config.database.get_mysql_yq_connection_string(),
        poolclass=sqlalchemy.pool.NullPool
    )
    
    end_date = datetime.now().strftime('%Y-%m-%d')
    start_date = (datetime.now() - timedelta(days=days)).strftime('%Y-%m-%d')
    
    query = f'''
        SELECT * FROM yq.cjqb 
        WHERE tradedDate >= '{start_date}' AND tradedDate <= '{end_date}'
    '''
    
    df = pd.read_sql(query, engine)
    
    # 数据预处理
    if not df.empty:
        df['tradedDate'] = pd.to_datetime(df['tradedDate'])
        df['dt'] = df['tradedDate'].dt.strftime('%Y-%m-%d')
        for col in ['tradedAmount', 'tradedPrice', 'tradedYield']:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
    
    print(f'加载 {len(df)} 条记录')
    return df

# 加载数据
df = load_trade_data(days=60)

## 3. 报告生成器类

In [ ]:
class TradeDataReportGenerator:
    """成交数据报告生成器"""
    
    def __init__(self, df: pd.DataFrame):
        self.df = df
        self.report_date = datetime.now().strftime('%Y-%m-%d')
    
    def generate_overview_section(self) -> Dict:
        """
        生成数据概览部分
        
        Returns:
            Dict: 概览数据
        """
        if self.df.empty:
            return {}
        
        overview = {
            '报告日期': self.report_date,
            '数据时间范围': {
                '开始日期': self.df['dt'].min(),
                '结束日期': self.df['dt'].max(),
                '天数': self.df['dt'].nunique()
            },
            '总体统计': {
                '总成交笔数': len(self.df),
                '总成交金额': float(self.df['tradedAmount'].sum()) if 'tradedAmount' in self.df.columns else 0,
                '成交债券数': int(self.df['bondCode'].nunique()) if 'bondCode' in self.df.columns else 0,
                '日均成交金额': float(self.df.groupby('dt')['tradedAmount'].sum().mean()) if 'tradedAmount' in self.df.columns else 0
            }
        }
        
        return overview
    
    def generate_liquidity_section(self) -> Dict:
        """
        生成流动性分析部分
        
        Returns:
            Dict: 流动性分析数据
        """
        if self.df.empty:
            return {}
        
        # 计算每日流动性指标
        daily_stats = self.df.groupby('dt').agg({
            'tradedAmount': 'sum',
            'bondCode': 'nunique'
        }).reset_index()
        
        daily_stats.columns = ['dt', 'amount', 'bond_count']
        
        # 计算移动平均
        daily_stats['ma5'] = daily_stats['amount'].rolling(5, min_periods=1).mean()
        daily_stats['ma20'] = daily_stats['amount'].rolling(20, min_periods=1).mean()
        
        latest = daily_stats.iloc[-1]
        
        liquidity = {
            '最新交易日': {
                '日期': latest['dt'],
                '成交金额': float(latest['amount']),
                '成交债券数': int(latest['bond_count']),
                '5日均线': float(latest['ma5']),
                '20日均线': float(latest['ma20'])
            },
            '流动性评估': {
                '趋势': '上升' if latest['amount'] > latest['ma5'] else '下降',
                '偏离度(%)': float((latest['amount'] - latest['ma5']) / latest['ma5'] * 100) if latest['ma5'] > 0 else 0
            }
        }
        
        return liquidity
    
    def generate_structure_section(self) -> Dict:
        """
        生成结构分析部分
        
        Returns:
            Dict: 结构分析数据
        """
        if self.df.empty:
            return {}
        
        structure = {}
        
        # 行业分布
        if 'yyIndustry' in self.df.columns:
            industry = self.df.groupby('yyIndustry')['tradedAmount'].sum().sort_values(ascending=False)
            total = industry.sum()
            structure['行业分布TOP10'] = [
                {
                    '行业': idx,
                    '成交金额': float(val),
                    '占比(%)': float(val / total * 100) if total > 0 else 0
                }
                for idx, val in industry.head(10).items()
            ]
        
        # 评级分布
        if 'yyRating' in self.df.columns:
            rating = self.df.groupby('yyRating')['tradedAmount'].sum().sort_values(ascending=False)
            total = rating.sum()
            structure['评级分布'] = [
                {
                    '评级': idx,
                    '成交金额': float(val),
                    '占比(%)': float(val / total * 100) if total > 0 else 0
                }
                for idx, val in rating.items()
            ]
        
        # 成交规模分布
        if 'tradedAmount' in self.df.columns:
            thresholds = config.trade_data.volume_thresholds
            bins = [0, thresholds['small'], thresholds['medium'], thresholds['large'], float('inf')]
            labels = ['小额', '中额', '大额', '超大额']
            
            df_temp = self.df.copy()
            df_temp['category'] = pd.cut(df_temp['tradedAmount'], bins=bins, labels=labels)
            
            amount_dist = df_temp.groupby('category')['tradedAmount'].agg(['count', 'sum'])
            total_count = amount_dist['count'].sum()
            total_amount = amount_dist['sum'].sum()
            
            structure['成交规模分布'] = [
                {
                    '类型': str(idx),
                    '笔数': int(row['count']),
                    '笔数占比(%)': float(row['count'] / total_count * 100) if total_count > 0 else 0,
                    '金额': float(row['sum']),
                    '金额占比(%)': float(row['sum'] / total_amount * 100) if total_amount > 0 else 0
                }
                for idx, row in amount_dist.iterrows()
            ]
        
        return structure
    
    def generate_top_bonds_section(self, top_n: int = 20) -> Dict:
        """
        生成活跃债券部分
        
        Args:
            top_n: 显示前N只债券
            
        Returns:
            Dict: 活跃债券数据
        """
        if self.df.empty:
            return {}
        
        # 按债券聚合
        bond_stats = self.df.groupby(['bondCode', 'bondName']).agg({
            'tradedAmount': ['sum', 'count'],
            'tradedYield': 'mean'
        }).reset_index()
        
        bond_stats.columns = ['bondCode', 'bondName', 'total_amount', 'trade_count', 'avg_yield']
        bond_stats = bond_stats.sort_values('total_amount', ascending=False).head(top_n)
        
        top_bonds = {
            '成交最活跃债券TOP{}'.format(top_n): [
                {
                    '债券代码': row['bondCode'],
                    '债券名称': row['bondName'],
                    '总成交金额': float(row['total_amount']),
                    '成交笔数': int(row['trade_count']),
                    '平均收益率(%)': float(row['avg_yield']) if pd.notna(row['avg_yield']) else None
                }
                for _, row in bond_stats.iterrows()
            ]
        }
        
        return top_bonds
    
    def generate_full_report(self) -> Dict:
        """
        生成完整报告
        
        Returns:
            Dict: 完整报告数据
        """
        report = {
            '报告标题': '债券成交数据分析报告',
            '报告生成时间': self.report_date,
            '数据概览': self.generate_overview_section(),
            '流动性分析': self.generate_liquidity_section(),
            '结构分析': self.generate_structure_section(),
            '活跃债券': self.generate_top_bonds_section()
        }
        
        return report

# 创建报告生成器实例
report_generator = TradeDataReportGenerator(df)
print('报告生成器初始化完成')

## 4. 生成并展示报告

In [ ]:
# 生成完整报告
report = report_generator.generate_full_report()

# 打印报告（格式化JSON）
print(json.dumps(report, ensure_ascii=False, indent=2))

## 5. 打印文本格式报告

In [ ]:
def print_text_report(report: Dict):
    """打印文本格式报告"""
    
    print('\n' + '='*70)
    print(report.get('报告标题', '成交数据分析报告'))
    print('='*70)
    print(f"报告生成时间: {report.get('报告生成时间', 'N/A')}")
    
    # 数据概览
    overview = report.get('数据概览', {})
    if overview:
        print('\n' + '-'*70)
        print('一、数据概览')
        print('-'*70)
        
        time_range = overview.get('数据时间范围', {})
        print(f"数据时间范围: {time_range.get('开始日期', 'N/A')} ~ {time_range.get('结束日期', 'N/A')}")
        print(f"交易日数: {time_range.get('天数', 'N/A')}")
        
        stats = overview.get('总体统计', {})
        print(f"\n总成交笔数: {stats.get('总成交笔数', 0):,}")
        print(f"总成交金额: {stats.get('总成交金额', 0)/1e8:.2f}亿元")
        print(f"成交债券数: {stats.get('成交债券数', 0):,}")
        print(f"日均成交金额: {stats.get('日均成交金额', 0)/1e8:.2f}亿元")
    
    # 流动性分析
    liquidity = report.get('流动性分析', {})
    if liquidity:
        print('\n' + '-'*70)
        print('二、流动性分析')
        print('-'*70)
        
        latest = liquidity.get('最新交易日', {})
        print(f"最新交易日: {latest.get('日期', 'N/A')}")
        print(f"成交金额: {latest.get('成交金额', 0)/1e8:.2f}亿元")
        print(f"成交债券数: {latest.get('成交债券数', 0)}")
        print(f"5日均线: {latest.get('5日均线', 0)/1e8:.2f}亿元")
        print(f"20日均线: {latest.get('20日均线', 0)/1e8:.2f}亿元")
        
        assessment = liquidity.get('流动性评估', {})
        print(f"\n趋势判断: {assessment.get('趋势', 'N/A')}")
        print(f"偏离度: {assessment.get('偏离度(%)', 0):.2f}%")
    
    # 结构分析
    structure = report.get('结构分析', {})
    if structure:
        print('\n' + '-'*70)
        print('三、结构分析')
        print('-'*70)
        
        # 行业分布
        industry = structure.get('行业分布TOP10', [])
        if industry:
            print('\n行业分布TOP5:')
            for item in industry[:5]:
                print(f"  {item['行业']}: {item['成交金额']/1e8:.2f}亿 ({item['占比(%)']:.1f}%)")
        
        # 评级分布
        rating = structure.get('评级分布', [])
        if rating:
            print('\n评级分布:')
            for item in rating[:8]:
                print(f"  {item['评级']}: {item['成交金额']/1e8:.2f}亿 ({item['占比(%)']:.1f}%)")
        
        # 成交规模分布
        amount_dist = structure.get('成交规模分布', [])
        if amount_dist:
            print('\n成交规模分布:')
            for item in amount_dist:
                print(f"  {item['类型']}: 笔数 {item['笔数占比(%)']:.1f}%, 金额 {item['金额占比(%)']:.1f}%")
    
    # 活跃债券
    top_bonds = report.get('活跃债券', {})
    if top_bonds:
        print('\n' + '-'*70)
        print('四、成交最活跃债券TOP10')
        print('-'*70)
        
        bonds = list(top_bonds.values())[0] if top_bonds else []
        for i, bond in enumerate(bonds[:10], 1):
            name = bond.get('债券名称', bond.get('债券代码', 'N/A'))
            amount = bond.get('总成交金额', 0) / 1e8
            count = bond.get('成交笔数', 0)
            print(f"{i}. {name}: {amount:.2f}亿 ({count}笔)")
    
    print('\n' + '='*70)
    print('报告结束')
    print('='*70)

# 打印文本格式报告
print_text_report(report)

## 6. 导出报告到文件

In [ ]:
def export_report_to_json(report: Dict, output_path: str = None) -> str:
    """
    导出报告为JSON文件
    
    Args:
        report: 报告数据
        output_path: 输出路径
        
    Returns:
        str: 文件路径
    """
    if output_path is None:
        output_path = f'trade_data_report_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json'
    
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
    
    print(f'报告已导出: {output_path}')
    return output_path

# 导出JSON报告
json_path = export_report_to_json(report)
print(f'JSON报告路径: {json_path}')

In [ ]:
def export_report_to_markdown(report: Dict, output_path: str = None) -> str:
    """
    导出报告为Markdown文件
    
    Args:
        report: 报告数据
        output_path: 输出路径
        
    Returns:
        str: 文件路径
    """
    if output_path is None:
        output_path = f'trade_data_report_{datetime.now().strftime("%Y%m%d_%H%M%S")}.md'
    
    lines = []
    
    # 标题
    lines.append(f"# {report.get('报告标题', '成交数据分析报告')}")
    lines.append(f"\n**报告生成时间**: {report.get('报告生成时间', 'N/A')}\n")
    
    # 数据概览
    overview = report.get('数据概览', {})
    if overview:
        lines.append('## 一、数据概览')
        
        time_range = overview.get('数据时间范围', {})
        lines.append(f"- **数据时间范围**: {time_range.get('开始日期', 'N/A')} ~ {time_range.get('结束日期', 'N/A')}")
        lines.append(f"- **交易日数**: {time_range.get('天数', 'N/A')}\n")
        
        stats = overview.get('总体统计', {})
        lines.append('| 指标 | 数值 |')
        lines.append('|------|------|')
        lines.append(f"| 总成交笔数 | {stats.get('总成交笔数', 0):,} |")
        lines.append(f"| 总成交金额 | {stats.get('总成交金额', 0)/1e8:.2f}亿元 |")
        lines.append(f"| 成交债券数 | {stats.get('成交债券数', 0):,} |")
        lines.append(f"| 日均成交金额 | {stats.get('日均成交金额', 0)/1e8:.2f}亿元 |\n")
    
    # 流动性分析
    liquidity = report.get('流动性分析', {})
    if liquidity:
        lines.append('## 二、流动性分析')
        
        latest = liquidity.get('最新交易日', {})
        lines.append(f"- **最新交易日**: {latest.get('日期', 'N/A')}")
        lines.append(f"- **成交金额**: {latest.get('成交金额', 0)/1e8:.2f}亿元")
        lines.append(f"- **成交债券数**: {latest.get('成交债券数', 0)}")
        lines.append(f"- **5日均线**: {latest.get('5日均线', 0)/1e8:.2f}亿元")
        lines.append(f"- **20日均线**: {latest.get('20日均线', 0)/1e8:.2f}亿元\n")
        
        assessment = liquidity.get('流动性评估', {})
        lines.append(f"**趋势判断**: {assessment.get('趋势', 'N/A')}")
        lines.append(f"**偏离度**: {assessment.get('偏离度(%)', 0):.2f}%\n")
    
    # 结构分析
    structure = report.get('结构分析', {})
    if structure:
        lines.append('## 三、结构分析')
        
        # 行业分布
        industry = structure.get('行业分布TOP10', [])
        if industry:
            lines.append('\n### 行业分布TOP5')
            lines.append('| 行业 | 成交金额 | 占比 |')
            lines.append('|------|----------|------|')
            for item in industry[:5]:
                lines.append(f"| {item['行业']} | {item['成交金额']/1e8:.2f}亿 | {item['占比(%)']:.1f}% |")
    
    # 活跃债券
    top_bonds = report.get('活跃债券', {})
    if top_bonds:
        lines.append('\n## 四、成交最活跃债券TOP10')
        lines.append('| 排名 | 债券名称 | 成交金额 | 成交笔数 |')
        lines.append('|------|----------|----------|----------|')
        
        bonds = list(top_bonds.values())[0] if top_bonds else []
        for i, bond in enumerate(bonds[:10], 1):
            name = bond.get('债券名称', bond.get('债券代码', 'N/A'))
            amount = bond.get('总成交金额', 0) / 1e8
            count = bond.get('成交笔数', 0)
            lines.append(f"| {i} | {name} | {amount:.2f}亿 | {count} |")
    
    # 写入文件
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))
    
    print(f'报告已导出: {output_path}')
    return output_path

# 导出Markdown报告
md_path = export_report_to_markdown(report)
print(f'Markdown报告路径: {md_path}')

## 7. 查看导出的报告内容

In [ ]:
# 读取并显示Markdown报告
with open(md_path, 'r', encoding='utf-8') as f:
    md_content = f.read()

print(md_content)